In [1]:
from src.graph.graph_pipeline import build_pipeline, run_pipeline

pipe = build_pipeline()
frage = "Nenne mir alle Messwerte der Epoche 3"
antwort = run_pipeline(pipe, frage)

print("Antwort des Systems:")
print(antwort)


[query_processor] input.query:
Nenne mir alle Messwerte der Epoche 3
[query_processor] output.cypher:
MATCH (s:Sensor)-[:GENERATED]->(m:Measurement)-[:EPOCH_IN]->(e:Epoch {epoch_id: 3})
RETURN m

[retriever] input.cypher:
MATCH (s:Sensor)-[:GENERATED]->(m:Measurement)-[:EPOCH_IN]->(e:Epoch {epoch_id: 3})
RETURN m
[retriever] output.documents_total: 30
[retriever] output.first_document:
{"m": {"light": 368.0, "temperature": 23.0362, "humidity": 38.4629, "time": "2004-03-12T09:32:29.519989", "voltage": 2.51661}}

[organizer] input.documents:
documents_total=30
[organizer] output.analysis_json preview (len=12576):
{"context": {"entity_count": 1, "entities": ["Unknown"], "provenance": "Cypher Query Result"}, "analysis": {"Unknown": {"temperature": {"stats_global": {"count": 30, "mean": 44.648046666666666, "std": 43.54458524484225, "min": {"value": 16.823, "timestamp": "2004-02-28T06:09:47.067740"}, "max": {"value": 122.153, "timestamp": "2004-03-17T13:02:24.900531"}, "quantiles": {"p05": 

In [ ]:
import csv
import json
import re
import os
from collections import Counter
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from neo4j import GraphDatabase

from tqdm import tqdm
from dotenv import load_dotenv

from haystack.components.evaluators import FaithfulnessEvaluator
from haystack.dataclasses import ChatMessage
from haystack_integrations.components.generators.ollama import OllamaChatGenerator

# Load environment variables
load_dotenv()

# -----------------------------
# Config
# -----------------------------
# Supports JSONL (one JSON per line) OR a JSON array file.
DATASET_PATH = Path("eval_dataset_with_gold.jsonl")

OLLAMA_URL = os.getenv("OLLAMA_URL", "http://ollama:11434")
GENERATOR_MODEL = os.getenv("OLLAMA_GENERATOR_MODEL") or os.getenv("OLLAMA_MODEL", "llama3.1:8b")
# Use the generator model or a specific judge model if defined
JUDGE_MODEL = os.getenv("OLLAMA_GENERATOR_MODEL") or os.getenv("OLLAMA_MODEL", "llama3.1:8b")

MAX_SAMPLES: Optional[int] = None
SHOW_PROGRESS = True

# Debug: store intermediate component outputs into JSON results
SAVE_COMPONENT_OUTPUTS = True
MAX_DEBUG_DOCS = 50

# Neo4j for Cypher validation (EXPLAIN)
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://neo4j:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")
_neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# -----------------------------
# Metrics
# -----------------------------
def _normalize(text: str) -> str:
    text = (text or "").lower().strip()
    text = re.sub(r"[^\w\s]+", " ", text)
    return " ".join(text.split())


def exact_match(pred: str, gold: str) -> bool:
    return _normalize(pred) == _normalize(gold)


def f1_score(pred: str, gold: str) -> float:
    pred_tokens = _normalize(pred).split()
    gold_tokens = _normalize(gold).split()
    if not pred_tokens or not gold_tokens:
        return 0.0

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


# -----------------------------
# IO helpers
# -----------------------------
def load_dataset(path: Path) -> List[Dict[str, Any]]:
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        return []

    # JSON array
    if text.startswith("["):
        data = json.loads(text)
        if not isinstance(data, list):
            raise ValueError("JSON array expected")
        return data

    # JSONL
    rows: List[Dict[str, Any]] = []
    for line in text.splitlines():
        line = line.strip()
        if line:
            rows.append(json.loads(line))
    return rows


def save_results_json(path: Path, payload: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def write_summary_csv(path: Path, rows: List[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        "id",
        "category",
        "question",
        "gold_answer",
        "prediction",
        "exact_match",
        "f1",
        "faithfulness",
        "faithfulness_error",
        "cypher",
        "cypher_policy_ok",
        "cypher_explain_ok",
        "cypher_error",
        "log_path",
        "generator_model",
        "judge_model",
    ]
    with path.open("w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({k: row.get(k, "") for k in fieldnames})


# -----------------------------
# Cypher checks
# -----------------------------
_FORBIDDEN = ["create", "merge", "delete", "set", "call", "load csv", "apoc", "drop", "remove"]


def cypher_policy_check(cypher: Optional[str]) -> Tuple[bool, Optional[str]]:
    if not cypher:
        return False, "empty cypher"
    s = cypher.strip()
    if "```" in s:
        return False, "code fence detected"
    if s.lower().startswith("cypher"):
        # some models prefix with 'Cypher:'
        s = s.split(":", 1)[-1].strip()
    low = s.lower()
    for kw in _FORBIDDEN:
        if kw in low:
            return False, f"forbidden keyword: {kw}"
    # single statement: reject semicolons except optional trailing
    if ";" in s.strip("; "):
        return False, "multiple statements / semicolon detected"
    if "return" not in low:
        return False, "missing RETURN"
    if not (low.startswith("match") or low.startswith("optional match")):
        return False, "query must start with MATCH"
    return True, None


def cypher_explain_ok(cypher: Optional[str]) -> Tuple[bool, Optional[str]]:
    ok, reason = cypher_policy_check(cypher)
    if not ok:
        return False, reason
    q = (cypher or "").strip()
    try:
        with _neo4j_driver.session() as session:
            session.run("EXPLAIN " + q).consume()
        return True, None
    except Exception as exc:
        return False, str(exc)


# -----------------------------
# Pipeline + judge
# -----------------------------
def run_pipeline_with_debug(pipe, question: str) -> Tuple[str, str, Dict[str, Any]]:
    """Runs the GraphRAG pipeline and (optionally) captures intermediate outputs."""
    include = ["query_processor", "retriever", "organizer", "generator"]
    result = pipe.run(
        {
            "query_processor": {"query": question},
            "generator": {"query": question},
        },
        include_outputs_from=include,
    )

    answer = result.get("generator", {}).get("answer", "")
    context = result.get("organizer", {}).get("context", "")

    debug: Dict[str, Any] = {
        "query_processor": {
            "cypher": result.get("query_processor", {}).get("cypher"),
        },
        "retriever": {
            "documents_total": None,
            "documents": None,
        },
        "organizer": {
            "context": context,
        },
        "generator": {
            "answer": answer,
            "model": GENERATOR_MODEL,
        },
    }

    docs = result.get("retriever", {}).get("documents")
    if isinstance(docs, list):
        debug["retriever"]["documents_total"] = len(docs)
        debug["retriever"]["documents"] = [
            {
                "content": getattr(d, "content", None),
                "meta": getattr(d, "meta", None),
            }
            for d in docs[:MAX_DEBUG_DOCS]
        ]

    return answer, context, debug


def _extract_json_object(text: Optional[str]) -> Optional[str]:
    if not text:
        return None
    text = text.strip()

    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            return json.dumps(parsed, ensure_ascii=False)
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None

    candidate = text[start : end + 1]
    try:
        parsed = json.loads(candidate)
        if isinstance(parsed, dict):
            return json.dumps(parsed, ensure_ascii=False)
    except Exception:
        return None

    return None


class JSONCoercingChatGenerator:
    """Small wrapper to keep FaithfulnessEvaluator robust if the judge adds extra text."""

    def __init__(self, base: OllamaChatGenerator):
        self._base = base

    def run(self, messages: List[ChatMessage]):
        res = self._base.run(messages=messages)
        replies = res.get("replies", [])
        if replies and replies[0].text:
            coerced = _extract_json_object(replies[0].text)
            if coerced is not None:
                replies[0] = ChatMessage.from_assistant(coerced)
        return {"replies": replies}


faithfulness_response_schema = {
    "type": "object",
    "properties": {
        "statements": {"type": "array", "items": {"type": "string"}},
        "statement_scores": {"type": "array", "items": {"type": "integer", "enum": [0, 1]}},
    },
    "required": ["statements", "statement_scores"],
}

judge_llm = OllamaChatGenerator(
    model=JUDGE_MODEL,
    url=OLLAMA_URL,
    generation_kwargs={"temperature": 0.0, "num_predict": 256},
    response_format=faithfulness_response_schema,
 )
faithfulness = FaithfulnessEvaluator(
    chat_generator=JSONCoercingChatGenerator(judge_llm),
    progress_bar=False,
    raise_on_failure=True,
 )


def faithfulness_single(
    *, question: str, context: str, predicted_answer: str, retries: int = 2
) -> Tuple[Optional[float], Optional[dict], Optional[str]]:
    last_err: Optional[Exception] = None
    for _ in range(retries + 1):
        try:
            res = faithfulness.run(
                questions=[question],
                contexts=[[context]],
                predicted_answers=[predicted_answer],
            )
            score = res.get("individual_scores", [None])[0]
            details = res.get("results", [None])[0]
            return score, details, None
        except Exception as exc:
            last_err = exc
    return None, None, str(last_err) if last_err else "unknown error"


# -----------------------------
# Evaluation
# -----------------------------
def evaluate_dataset(
    *,
    pipe,
    dataset_path: Path,
    max_samples: Optional[int] = None,
    show_progress: bool = True,
) -> Dict[str, Any]:
    data = load_dataset(dataset_path)
    if max_samples is not None:
        data = data[:max_samples]

    results: List[Dict[str, Any]] = []
    run_id = datetime.utcnow().strftime("%Y%m%dT%H%M%S")
    logs_dir = Path("outputs") / "logs" / run_id
    logs_dir.mkdir(parents=True, exist_ok=True)

    iterator = tqdm(data, disable=not show_progress, desc="Evaluating")
    for idx, row in enumerate(iterator):
        q = str(row.get("question", row.get("query", "")))
        gold = str(row.get("gold_answer", ""))

        component_outputs: Optional[Dict[str, Any]] = None
        cypher = None
        try:
            pred, ctx, dbg = run_pipeline_with_debug(pipe, q)
            cypher = (dbg.get("query_processor") or {}).get("cypher")
            if SAVE_COMPONENT_OUTPUTS:
                component_outputs = dbg
        except Exception as exc:
            pred, ctx = f"[ERROR] {exc}", ""
            if SAVE_COMPONENT_OUTPUTS:
                component_outputs = {"error": str(exc)}

        # Cypher validation (policy + EXPLAIN)
        cy_policy_ok, cy_policy_err = cypher_policy_check(cypher)
        cy_explain_ok, cy_explain_err = cypher_explain_ok(cypher)
        cy_err = cy_policy_err if not cy_policy_ok else cy_explain_err

        em = exact_match(pred, gold)
        f1 = f1_score(pred, gold)

        f_score, f_details, f_err = faithfulness_single(
            question=q,
            context=ctx,
            predicted_answer=pred,
            retries=2,
        )

        log_path = logs_dir / f"{idx:04d}_{row.get('id', 'sample')}.json"
        result_row: Dict[str, Any] = {
            "id": row.get("id"),
            "category": row.get("category"),
            "question": q,
            "gold_answer": gold,
            "prediction": pred,
            "context": ctx,
            "exact_match": em,
            "f1": f1,
            "faithfulness": f_score,
            "faithfulness_details": f_details,
            "faithfulness_error": f_err,
            "cypher": cypher,
            "cypher_policy_ok": cy_policy_ok,
            "cypher_explain_ok": cy_explain_ok,
            "cypher_error": cy_err,
            "log_path": str(log_path),
            "generator_model": GENERATOR_MODEL,
            "judge_model": JUDGE_MODEL,
        }
        if SAVE_COMPONENT_OUTPUTS:
            result_row["component_outputs"] = component_outputs

        results.append(result_row)
        if SAVE_COMPONENT_OUTPUTS:
            save_results_json(log_path, result_row)

    valid_faith = [r["faithfulness"] for r in results if isinstance(r.get("faithfulness"), (int, float))]
    avg_faith = float(sum(valid_faith) / len(valid_faith)) if valid_faith else 0.0

    cy_valid = [r for r in results if r.get("cypher_explain_ok") is True]
    cy_rate = float(len(cy_valid) / len(results)) if results else 0.0

    summary = {
        "run_id": run_id,
        "samples": len(results),
        "exact_match": round(sum(1 for r in results if r.get("exact_match")) / len(results), 4) if results else 0.0,
        "avg_f1": round(sum(float(r.get("f1", 0.0)) for r in results) / len(results), 4) if results else 0.0,
        "faithfulness_score": round(avg_faith, 4),
        "cypher_explain_ok_rate": round(cy_rate, 4),
        "judge_model": JUDGE_MODEL,
        "generator_model": GENERATOR_MODEL,
        "ollama_url": OLLAMA_URL,
        "dataset": str(dataset_path),
        "save_component_outputs": SAVE_COMPONENT_OUTPUTS,
        "max_debug_docs": MAX_DEBUG_DOCS,
        "logs_dir": str(logs_dir),
    }

    test_path = logs_dir / "results.json"
    save_results_json(test_path, {"summary": summary, "results": results})

    legacy_path = Path("outputs") / "test.json"
    save_results_json(legacy_path, {"summary": summary, "results": results})

    csv_path = Path("outputs") / f"summary_{run_id}.csv"
    write_summary_csv(csv_path, results)

    return {"summary": summary, "json_path": str(test_path), "csv_path": str(csv_path), "logs_dir": str(logs_dir)}


evaluation = evaluate_dataset(
    pipe=pipe,
    dataset_path=DATASET_PATH,
    max_samples=MAX_SAMPLES,
    show_progress=SHOW_PROGRESS,
 )

print("Summary:")
print(evaluation["summary"])
print("Saved JSON:", evaluation["json_path"])
print("Saved CSV:", evaluation["csv_path"])
print("Logs dir:", evaluation["logs_dir"])

PromptBuilder has 3 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.
Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]


[query_processor] input.query:
Welche Temperatur hat Sensor 1.0 am 2004-02-28 um 00:59:16 gemessen?
[query_processor] output.cypher:
MATCH (s:Sensor {mote_id: 1.0})-[:GENERATED]->(m:Measurement)-[:AT_DATE]->(d:Date {date: "2004-02-28"})-[:HAS_HOUR]->(h:Hour {hour: 0})-[:HAS_MINUTE]->(mi:Minute {minute: 59})
RETURN DISTINCT m

[retriever] input.cypher:
MATCH (s:Sensor {mote_id: 1.0})-[:GENERATED]->(m:Measurement)-[:AT_DATE]->(d:Date {date: "2004-02-28"})-[:HAS_HOUR]->(h:Hour {hour: 0})-[:HAS_MINUTE]->(mi:Minute {minute: 59})
RETURN DISTINCT m
[query_processor] output.cypher:
MATCH (s:Sensor {mote_id: 1.0})-[:GENERATED]->(m:Measurement)-[:AT_DATE]->(d:Date {date: "2004-02-28"})-[:HAS_HOUR]->(h:Hour {hour: 0})-[:HAS_MINUTE]->(mi:Minute {minute: 59})
RETURN DISTINCT m

[retriever] input.cypher:
MATCH (s:Sensor {mote_id: 1.0})-[:GENERATED]->(m:Measurement)-[:AT_DATE]->(d:Date {date: "2004-02-28"})-[:HAS_HOUR]->(h:Hour {hour: 0})-[:HAS_MINUTE]->(mi:Minute {minute: 59})
RETURN DISTINCT m
[re

Evaluating:   1%|          | 1/100 [00:09<16:12,  9.83s/it]


[query_processor] input.query:
Welche Luftfeuchtigkeit hat Sensor 1.0 am 2004-02-28 um 01:03:16 gemessen?
[query_processor] output.cypher:
MATCH (s:Sensor {mote_id: 1.0})-[:GENERATED]->(m:Measurement)-[:AT_DATE]->(d:Date {date: "2004-02-28"})-[:HAS_HOUR]->(h:Hour {hour: 1})-[:HAS_MINUTE]->(mi:Minute {minute: 3})
RETURN m.voltage AS Luftfeuchtigkeit

[retriever] input.cypher:
MATCH (s:Sensor {mote_id: 1.0})-[:GENERATED]->(m:Measurement)-[:AT_DATE]->(d:Date {date: "2004-02-28"})-[:HAS_HOUR]->(h:Hour {hour: 1})-[:HAS_MINUTE]->(mi:Minute {minute: 3})
RETURN m.voltage AS Luftfeuchtigkeit
[query_processor] output.cypher:
MATCH (s:Sensor {mote_id: 1.0})-[:GENERATED]->(m:Measurement)-[:AT_DATE]->(d:Date {date: "2004-02-28"})-[:HAS_HOUR]->(h:Hour {hour: 1})-[:HAS_MINUTE]->(mi:Minute {minute: 3})
RETURN m.voltage AS Luftfeuchtigkeit

[retriever] input.cypher:
MATCH (s:Sensor {mote_id: 1.0})-[:GENERATED]->(m:Measurement)-[:AT_DATE]->(d:Date {date: "2004-02-28"})-[:HAS_HOUR]->(h:Hour {hour: 1})-[

Evaluating:   2%|▏         | 2/100 [00:14<11:17,  6.92s/it]

...


[query_processor] input.query:
Welche Temperatur hat Sensor 3.0 am 2004-02-28 um 01:35:16 gemessen?
[query_processor] output.cypher:
MATCH (s:Sensor {mote_id: 3.0})-[:GENERATED]->(m:Measurement)-[:AT_DATE]->(d:Date {date: "2004-02-28"})-[:HAS_HOUR]->(h:Hour {hour: 1})-[:HAS_MINUTE]->(mi:Minute {minute: 35})
RETURN DISTINCT m

[retriever] input.cypher:
MATCH (s:Sensor {mote_id: 3.0})-[:GENERATED]->(m:Measurement)-[:AT_DATE]->(d:Date {date: "2004-02-28"})-[:HAS_HOUR]->(h:Hour {hour: 1})-[:HAS_MINUTE]->(mi:Minute {minute: 35})
RETURN DISTINCT m
[retriever] output.documents_total: 1082
[retriever] output.first_document:
{"m": {"light": 46.92, "temperature": 17.8814, "humidity": 39.3483, "time": "2004-02-28T05:40:17.117496", "voltage": 2.67532}}

[organizer] input.documents:
documents_total=1082
[organizer] output.analysis_json preview (len=64565):
{"context": {"entity_count": 1, "entities": ["Unknown"], "provenance": "Cypher Query Result"}, "analysis": {"Unknown": {"temperature": {"stats_

Evaluating:   9%|▉         | 9/100 [00:59<10:01,  6.61s/it]


[query_processor] input.query:
Welche Luftfeuchtigkeit hat Sensor 3.0 am 2004-02-28 um 01:40:16 gemessen?
[query_processor] output.cypher:
MATCH (s:Sensor {mote_id: 3.0})-[:GENERATED]->(m:Measurement)-[:AT_DATE]->(d:Date {date: "2004-02-28"})-[:HAS_HOUR]->(h:Hour {hour: 1})-[:HAS_MINUTE]->(mi:Minute {minute: 40})
RETURN m.voltage AS Luftfeuchtigkeit

[retriever] input.cypher:
MATCH (s:Sensor {mote_id: 3.0})-[:GENERATED]->(m:Measurement)-[:AT_DATE]->(d:Date {date: "2004-02-28"})-[:HAS_HOUR]->(h:Hour {hour: 1})-[:HAS_MINUTE]->(mi:Minute {minute: 40})
RETURN m.voltage AS Luftfeuchtigkeit
[retriever] output.documents_total: 1082
[retriever] output.first_document:
{"Luftfeuchtigkeit": 2.67532}

[organizer] input.documents:
documents_total=1082
[organizer] output.analysis_json preview (len=7137):
{"context": {"entity_count": 1, "entities": ["Unknown"], "provenance": "Cypher Query Result"}, "analysis": {"Unknown": {"Luftfeuchtigkeit": {"stats_global": {"count": 1082, "mean": 2.69813002772643

Evaluating:  10%|█         | 10/100 [01:06<09:53,  6.59s/it]


[query_processor] input.query:
Welche Helligkeit hat Sensor 3.0 am 2004-02-28 um 01:45:16 gemessen?
[query_processor] output.cypher:
MATCH (s:Sensor {mote_id: 3.0})-[:GENERATED]->(m:Measurement)-[:AT_DATE]->(d:Date {date: "2004-02-28"})-[:HAS_HOUR]->(h:Hour {hour: 1})-[:HAS_MINUTE]->(mi:Minute {minute: 45})
RETURN m.light

[retriever] input.cypher:
MATCH (s:Sensor {mote_id: 3.0})-[:GENERATED]->(m:Measurement)-[:AT_DATE]->(d:Date {date: "2004-02-28"})-[:HAS_HOUR]->(h:Hour {hour: 1})-[:HAS_MINUTE]->(mi:Minute {minute: 45})
RETURN m.light
[retriever] output.documents_total: 1082
[retriever] output.first_document:
{"m.light": 46.92}

[organizer] input.documents:
documents_total=1082
[organizer] output.analysis_json preview (len=5046):
{"context": {"entity_count": 1, "entities": ["Unknown"], "provenance": "Cypher Query Result"}, "analysis": {"Unknown": {"light": {"stats_global": {"count": 1082, "mean": 208.1215157116451, "std": 149.01882457346272, "min": {"value": 46.92, "timestamp": "2025